<a href="https://colab.research.google.com/github/keyvanmahmoudi/LLM_Engineering_Course/blob/main/solutions/Session08_building_a_rag_system_solutions.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

# پاسخ تمرین‌های جلسهٔ ۸: ساخت سیستم RAG

این نوت‌بوک پاسخ پیشنهادی تمرین‌های انتهای [نوت‌بوک جلسه](../sessions/Session08_building_a_rag_system.ipynb) است. قبل از دیدن پاسخ‌ها، خودتان تمرین‌ها را حل کنید؛ معمولاً بیش از یک راه‌حل درست وجود دارد.

</div>

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

### آماده‌سازی

</div>

In [ ]:
!pip -q install openai sentence-transformers hazm
!wget -q -N https://raw.githubusercontent.com/keyvanmahmoudi/LLM_Engineering_Course/main/llm_utils.py

from llm_utils import setup_llm, chat

client, MODEL = setup_llm("ollama")      # or: setup_llm("api")

In [ ]:
import json
import re

import numpy as np
from hazm import Normalizer
from llm_utils import download_data
from sentence_transformers import SentenceTransformer

chunks = [json.loads(line) for line in open(download_data("novinkala_chunks.jsonl"), encoding="utf-8")]
_normalizer = Normalizer(persian_numbers=False)
_DIGITS = str.maketrans("۰۱۲۳۴۵۶۷۸۹٠١٢٣٤٥٦٧٨٩", "01234567890123456789")


def normalize(text):
    return _normalizer.normalize(text).translate(_DIGITS)


class Retriever:
    def __init__(self, chunks, model_name="intfloat/multilingual-e5-base"):
        self.chunks = chunks
        self.model = SentenceTransformer(model_name)
        self.vectors = self.model.encode([f"passage: {c['text']}" for c in chunks], normalize_embeddings=True)

    def search(self, query, k=4):
        q = self.model.encode([f"query: {normalize(query)}"], normalize_embeddings=True)[0]
        scores = self.vectors @ q
        top = np.argsort(-scores)[:k]
        return [{**self.chunks[i], "score": float(scores[i])} for i in top]


retriever = Retriever(chunks)

REFUSAL = "متأسفانه در اسناد نوین‌کالا پاسخی برای این پرسش پیدا نکردم."
RAG_SYSTEM = f"""You are the customer-support assistant of Novin Kala, an Iranian online store.
Answer the customer's question in Persian, using ONLY the numbered sources provided.
Rules:
- Every factual sentence must end with the number of its source in brackets, e.g. [2].
- If the sources do not contain the answer, reply exactly: «{REFUSAL}»
- Do not use outside knowledge and do not guess numbers, prices or dates.
- Mention conditions and exceptions that apply. Be concise and polite."""


def build_prompt(question, hits):
    sources = "\n\n".join(f"<source id=\"{i}\">\n{h['text']}\n</source>" for i, h in enumerate(hits, 1))
    return f"{sources}\n\nCustomer question: {question}"


def rag_answer(question, system=RAG_SYSTEM, k=4):
    hits = retriever.search(question, k=k)
    return chat(client, MODEL, build_prompt(question, hits), system=system, temperature=0), hits

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

### تمرین ۱

قاعده امتناع را از `RAG_SYSTEM` حذف کنید و پرسش‌های بخش ۵ را دوباره اجرا کنید. چه تغییری می‌بینید؟

</div>

In [ ]:
HARD_QUESTIONS = [
    "قیمت دلار امروز چنده؟",
    "هزینه ارسال سریع به تبریز چقدر است؟",
    "اگر یخچال را قسطی بخرم، پرداخت در محل هم دارد؟",
    "رمزی که پیامک میشه تا کی اعتبار داره؟",
]

NO_REFUSAL_SYSTEM = "\n".join(line for line in RAG_SYSTEM.split("\n") if "do not contain" not in line)

for q in HARD_QUESTIONS:
    with_rule, _ = rag_answer(q)
    without_rule, _ = rag_answer(q, system=NO_REFUSAL_SYSTEM)
    print(f"Q: {q}\n  with refusal rule   : {with_rule}\n  without refusal rule: {without_rule}\n")

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

بدون قاعدهٔ امتناع، مدل برای پرسش خارج از دامنه، مثل قیمت دلار، معمولاً یا از دانش خودش پاسخ می‌دهد، یا عددی ساختگی می‌گوید، یا جمله‌ای مبهم می‌نویسد. پرسش با پیش‌فرض غلط، یعنی ارسال سریع به تبریز، از همه خطرناک‌تر است: منابع بازیابی‌شده دربارهٔ ارسال سریع و هزینهٔ ۸۵ هزار تومانی‌اند و مدل بدون قاعدهٔ صریح، به‌احتمال زیاد همین عدد را برای تبریز هم می‌گوید.

قاعدهٔ «Do not use outside knowledge» هنوز در پرامپت هست، ولی بدون مشخص کردن **رفتار جایگزین**، مدل نمی‌داند وقتی جواب نیست چه کند. این یک الگوی کلی پرامپت‌نویسی است: به جای فقط «نکن»، بگویید «به جایش این کار را بکن».

</div>

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

### تمرین ۲

تابع `rag_answer` را طوری تغییر دهید که اگر امتیاز بهترین تکه کمتر از یک آستانه بود، بدون فراخواندن مدل امتناع کند. آستانه مناسب را با چند پرسش داخل و خارج از دامنه پیدا کنید.

</div>

In [ ]:
in_domain = [json.loads(line)["question"] for line in open(download_data("rag_eval.jsonl"), encoding="utf-8")]
out_of_domain = ["قیمت دلار امروز چنده؟", "بهترین رستوران شیراز کجاست؟", "فرمول مساحت دایره چیه؟",
                 "نوین‌کالا شعبه حضوری در شیراز دارد؟", "برای خرید با ارز دیجیتال چه کنم؟",
                 "هوای فردا تهران چطوره؟", "پایتون یا جاوا؟", "آیا لپ‌تاپ دست دوم خرید می‌کنید؟"]

top_in = [retriever.search(q, k=1)[0]["score"] for q in in_domain]
top_out = [retriever.search(q, k=1)[0]["score"] for q in out_of_domain]
print(f"in-domain     min={min(top_in):.3f}  mean={np.mean(top_in):.3f}")
print(f"out-of-domain max={max(top_out):.3f}  mean={np.mean(top_out):.3f}")

import matplotlib.pyplot as plt
plt.hist(top_in, bins=15, alpha=0.6, label="in-domain")
plt.hist(top_out, bins=15, alpha=0.6, label="out-of-domain")
plt.xlabel("best chunk score")
plt.legend()
plt.show()

In [ ]:
THRESHOLD = (min(top_in) + max(top_out)) / 2        # adjust after looking at the histogram
print(f"threshold = {THRESHOLD:.3f}")


def guarded_answer(question, k=4):
    hits = retriever.search(question, k=k)
    if hits[0]["score"] < THRESHOLD:
        return REFUSAL                                  # no model call: cheaper and safer
    return chat(client, MODEL, build_prompt(question, hits), system=RAG_SYSTEM, temperature=0)


for q in ["قیمت دلار امروز چنده؟", "هزینه ارسال عادی چقدر است؟"]:
    print(f"Q: {q}\nA: {guarded_answer(q)}\n")

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

در E5 امتیازها در بازهٔ باریکی جمع می‌شوند، معمولاً بین ۰٫۷ و ۰٫۹. پس دو توزیع ممکن است هم‌پوشانی داشته باشند و آستانهٔ کامل وجود نداشته باشد. پرسش‌هایی مثل «شعبه حضوری در شیراز» از نظر واژگان به اسناد نزدیک‌اند، چون دربارهٔ نوین‌کالا و شیرازند، ولی پاسخشان در اسناد نیست. این پرسش‌ها از آستانه عبور می‌کنند و فقط قاعدهٔ امتناع مدل جلویشان را می‌گیرد.

پس آستانه **جایگزین** قاعدهٔ امتناع نیست، **لایهٔ اول** دفاع است. پرسش‌های کاملاً بی‌ربط را بدون هزینهٔ فراخوانی مدل رد می‌کند. آستانه را محافظه‌کارانه، یعنی پایین، انتخاب کنید تا هیچ پرسش درستی رد نشود. ردکردن اشتباه پرسش درست برای مشتری بدتر از یک فراخوانی اضافهٔ مدل است.

</div>

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

### تمرین ۳

یک «بررسی ارجاع» بسازید: برای هر جمله پاسخ که ارجاع دارد، از مدل بپرسید آیا منبع ارجاع‌داده‌شده آن جمله را پشتیبانی می‌کند یا نه.

</div>

In [ ]:
from pydantic import BaseModel


class Support(BaseModel):
    supported: bool
    reason: str


CHECK_SYSTEM = ("You verify citations. Decide whether the SOURCE fully supports the CLAIM. "
                "A claim is supported only if every fact in it (numbers, conditions, names) is stated "
                "in the source. Give a short reason.")


def check_citations(answer, hits):
    report = []
    for sentence in re.split(r"(?<=[.!؟?])\s+", answer):
        for n in {int(x) for x in re.findall(r"\[(\d+)\]", sentence)}:
            if not 0 < n <= len(hits):
                report.append((sentence, n, False, "citation number out of range"))
                continue
            claim = re.sub(r"\s*\[\d+\]", "", sentence)
            response = client.chat.completions.create(
                model=MODEL, temperature=0,
                messages=[{"role": "system", "content": CHECK_SYSTEM},
                          {"role": "user", "content": f"SOURCE:\n{hits[n - 1]['text']}\n\nCLAIM: {claim}"}],
                response_format={"type": "json_schema",
                                 "json_schema": {"name": "Support", "schema": Support.model_json_schema()}})
            verdict = Support.model_validate_json(response.choices[0].message.content)
            report.append((sentence, n, verdict.supported, verdict.reason))
    return report


answer, hits = rag_answer("شرایط بازگشت هدفون داخل گوشی چیست و پولم کی برمی‌گردد؟")
print(answer, "\n")
for sentence, n, ok, reason in check_citations(answer, hits):
    print(f"[{n}] {'OK ' if ok else 'BAD'} {sentence}\n      {reason}")

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

این بررسی هر جمله را با **همان منبعی** که به آن ارجاع داده مقایسه می‌کند، نه با همهٔ منابع. پس دو نوع خطا را می‌گیرد: ادعای ساختگی، و ادعای درست با ارجاع اشتباه. هزینه‌اش یک فراخوانی مدل برای هر ارجاع است. پس معمولاً آن را برای پاسخ‌های حساس اجرا می‌کنند، یا در ارزیابی آفلاین روی نمونه‌ای از گفتگوها.

جمله‌هایی که اصلاً ارجاع ندارند هم بررسی نمی‌شوند. اگر قاعدهٔ «هر جملهٔ واقعیت‌محور ارجاع دارد» رعایت نشده باشد، این جمله‌ها خودشان نشانهٔ خطرند و می‌توان آن‌ها را جداگانه گزارش کرد.

</div>

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

### تمرین ۴

اسناد یک حوزه دیگر، مثلاً آیین‌نامه آموزشی یا راهنمای یک نرم‌افزار، را با خط لوله جلسه ۷ آماده کنید و با همین کد یک دستیار RAG برای آن بسازید.

</div>

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

اسناد خود را، مثلاً چند فایل Word یا Markdown از یک آیین‌نامه، بارگذاری کنید. سلول زیر اسناد را می‌خواند، با روش جلسهٔ ۷ تقسیم می‌کند و یک بازیاب تازه می‌سازد. بقیهٔ کد RAG بدون تغییر کار می‌کند.

</div>

In [ ]:
!pip -q install python-docx
from docx import Document

try:
    from google.colab import files
    paths = list(files.upload())
except ImportError:
    paths = ["my_rules.docx"]          # change to your own files


def read_any(path):
    if path.lower().endswith(".docx"):
        return "\n\n".join(p.text for p in Document(path).paragraphs if p.text.strip())
    return open(path, encoding="utf-8").read()


def simple_chunks(text, source, size=600):
    paragraphs, chunks, current = [p.strip() for p in text.split("\n\n") if p.strip()], [], ""
    for p in paragraphs:
        if current and len(current) + len(p) > size:
            chunks.append({"source": source, "title": source, "section": "", "text": current.strip()})
            current = ""
        current += "\n" + p
    if current.strip():
        chunks.append({"source": source, "title": source, "section": "", "text": current.strip()})
    return chunks


my_chunks = [c for path in paths for c in simple_chunks(normalize(read_any(path)), path)]
retriever = Retriever(my_chunks)
RAG_SYSTEM = RAG_SYSTEM.replace("Novin Kala, an Iranian online store", "the organisation that wrote these documents")
REFUSAL_OWN = "متأسفانه در اسناد پاسخی برای این پرسش پیدا نکردم."
RAG_SYSTEM = RAG_SYSTEM.replace(REFUSAL, REFUSAL_OWN)
print(len(my_chunks), "chunks")

answer, hits = rag_answer("یک پرسش دربارهٔ اسناد خودتان بنویسید", system=RAG_SYSTEM)
print(answer)

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

هنگام ساخت دستیار برای حوزهٔ جدید، پیش از هر چیز ده تا بیست پرسش با پاسخ درست بنویسید و همان آزمون‌های دشوار جلسه را اجرا کنید: پرسش خارج از دامنه، پیش‌فرض غلط، پرسش ترکیبی و محاوره‌ای. در اسناد حقوقی و آیین‌نامه‌ها، ارجاع به منبع اهمیت بیشتری دارد، چون کاربر باید بتواند متن اصلی ماده را ببیند. می‌توانید شمارهٔ ماده یا صفحه را در متادیتای هر تکه نگه دارید و در پاسخ نمایش دهید.

</div>